In [ ]:
# Cell 1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.config import DATA_INTERIM, DATA_PROCESSED

surfaces = np.load(DATA_INTERIM / "interpolation_surfaces.npz")
metrics = pd.read_csv(DATA_INTERIM / "validation_metrics.csv")
print(metrics)

In [ ]:
# Cell 2 — side-by-side IDW vs Kriging
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
extent = [
    surfaces["grid_lon"].min(), surfaces["grid_lon"].max(),
    surfaces["grid_lat"].min(), surfaces["grid_lat"].max(),
]

im1 = axes[0].imshow(surfaces["idw_ws10"], extent=extent, origin="lower",
                     cmap="viridis", aspect="auto")
axes[0].set_title("IDW — wind speed at 10m (m/s)")
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(surfaces["krg_ws10"], extent=extent, origin="lower",
                     cmap="viridis", aspect="auto")
axes[1].set_title("Kriging — wind speed at 10m (m/s)")
plt.colorbar(im2, ax=axes[1])

# Overlay station locations
stations = pd.read_parquet(DATA_PROCESSED / "stations_summary.parquet")
for ax in axes:
    ax.scatter(stations["lon"], stations["lat"], c="red",
               s=30, marker="^", edgecolor="black", label="NOAA stations")
    ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
    ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Cell 3 — uncertainty surface from Kriging (the killer plot)
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(np.sqrt(surfaces["krg_var10"]), extent=extent, origin="lower",
               cmap="RdYlGn_r", aspect="auto")
ax.scatter(stations["lon"], stations["lat"], c="black",
           s=30, marker="^", label="Known stations")
ax.set_title("Kriging prediction uncertainty (std dev, m/s)")
plt.colorbar(im, label="Uncertainty")
ax.legend(); plt.show()
# ^ This plot shows uncertainty is LOW near stations, HIGH in data-sparse zones.
# This is the kind of plot that makes interviewers nod.

In [ ]:
# Cell 4 — validation scatter
val = pd.read_csv(DATA_INTERIM / "station_validation.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, method in zip(axes, ["idw", "krg"]):
    ax.scatter(val["actual"], val[f"{method}_pred"], alpha=0.6)
    lims = [val["actual"].min() - 0.5, val["actual"].max() + 0.5]
    ax.plot(lims, lims, "r--", label="1:1 line")
    ax.set_xlabel("Actual (NOAA)"); ax.set_ylabel(f"{method.upper()} predicted")
    ax.set_title(f"{method.upper()} validation")
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()